<a href="https://colab.research.google.com/github/shrikantss1/AgenticAI/blob/main/CRM_Lead_Qualifier_Agent_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM Lead Qualifier Agent

In [ ]:
import os
import json
from google.colab import userdata
from openai import OpenAI

In [ ]:
# initialize the OpenAI Client

try:
  client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
  print(f"Error: {e}")

In [ ]:
def lookup_domain_info(domain:str) -> str:
  """
  Looks up and returns the company information based on its domain.
  """

  print(f"Tool ACTIVATED: Looking for domain info {domain}")

  # Mock database for demonstration
  mock_data = {
      "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
      "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
      "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
  }

  info = mock_data.get(domain)
  if info:
    return json.dumps(info)
  else:
    return "Domain not found in the mock database."


In [ ]:
def check_crm_history(email: str) -> str:
  """
  Check the CRM data for the past engagement history with the lead.
  """

  print(f"Tool ACTIVATED: Checking CRM history for {email}")
  mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }

  info = mock_data.get(email, mock_data["default"])
  if info:
    return json.dumps(info)
  else:
    return "Email not found in the mock database."


In [ ]:
def calculate_lead_score(data_summary: str) -> str:
  """
  Analyzes the data summary (domain and CRM history) and calculates a lead score (High/Medium/Low).
  """
  print(f"Tool ACTIVATED: Calculate the Lead Score {data_summary}")
  data = json.loads(data_summary)
  score = "Low"
  # Simple scoring logic for demonstration
  if data["domain_info"].get("revenue", "").startswith("$1B+"):
      score = "High"
  elif data["crm_history"].get("status") == "Active Opportunity":
      score = "High"
  elif data["domain_info"].get("revenue", "").startswith("$50M"):
      score = "Medium"

  return json.dumps({"lead_score": score})

In [ ]:
AVAILABLE_FUNCTIONS = {
    "lookup_domain_info": lookup_domain_info,
    "check_crm_history": check_crm_history,
    "calculate_lead_score": calculate_lead_score,
}

In [ ]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info",
            "description": "Looks up and returns the company information based on its domain.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {
                        "type": "string",
                        "description": "The domain name of the company (e.g., 'acmecorp.com')"
                    }
                },
                "required": ["domain"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history",
            "description": "Checks the CRM data for past engagement history with the lead.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "The email address of the lead (e.g., 'jane@acmecorp.com')"
                    }
                },
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score",
            "description": "Analyzes the data summary (domain and CRM history) and calculates a lead score (High/Medium/Low).",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {
                        "type": "string",
                        "description": "A JSON string containing the summarized data, including 'domain_info' and 'crm_history' (e.g., '{\"domain_info\": {\"industry\": \"Software/SaaS\", \"size\": \"501-1000 employees\", \"revenue\": \"$50M - $100M\"}, \"crm_history\": {\"last_contact\": \"2025-11-15\", \"status\": \"Cold Lead\", \"notes\": \"Attended webinar, no follow-up yet.\"}}')"
                    }
                },
                "required": ["data_summary"]
            }
        }
    }
]

In [ ]:
def run_agent(user_prompt: str):
  """
  The main execution loop for the CRM lead qualifier agent.
  """
  print("\n Running the qualifier Agent")
  system_prompt = """
  You are an AI assistant designed to qualify CRM leads by gathering relevant information and calculating a lead score. You have access to the following tools:

  1.  **lookup_domain_info(domain: str)**:
  2.  **check_crm_history(email: str)**:
  3.  **calculate_lead_score(data_summary: str)**:

  Your goal is to:
  1.  Extract the lead's email and domain from the user's prompt.
  2.  Sequentially use `lookup_domain_info` to get company details and `check_crm_history` to retrieve past engagement data.
  3.  Combine the information from both tools into a `data_summary`.
  4.  Use `calculate_lead_score` with the `data_summary` to determine the lead's score.
  5.  Finally, provide a comprehensive summary of the lead, including all gathered information and the calculated lead score.

  Always prioritize using the available tools to gather factual information before making any conclusions or providing the final summary.
  """

  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]



  collected_data = {}

  while True:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools_schema,
        tool_choice="auto",
    )
    response_message = response.choices[0].message

    messages.append(response_message)

    tool_calls = response_message.tool_calls
    if tool_calls:
      for tool_call in tool_calls:
        function_name = tool_call.function.name
        function_to_call = AVAILABLE_FUNCTIONS[function_name]
        function_args = json.loads(tool_call.function.arguments)


        # Update the persistent memory (collected_data)
        if function_name == "lookup_domain_info":
            function_result = function_to_call(**function_args)
            collected_data["domain_info"] = json.loads(function_result)
        elif function_name == "check_crm_history":
            function_result = function_to_call(**function_args)
            collected_data["crm_history"] = json.loads(function_result)
        elif function_name == "calculate_lead_score":
            # Inject the accumulated data from previous turns
            function_args = {"data_summary": json.dumps(collected_data)}
            function_result = function_to_call(**function_args)

        messages.append(
            {
                "role": "tool",
                "content": function_result,
                "tool_call_id": tool_call.id,
                "name": function_name,
            }
        )
    else:
      print("\n--- FINAL AGENT SUMMARY ---")
      print(response_message.content)
      break



In [ ]:
lead_email_1 = "jane@acmecorp.com"
run_agent(f"Please qualify this lead for my call tomorrow: {lead_email_1}")

print("\n" + "="*80 + "\n")



 Running the qualifier Agent
Tool ACTIVATED: Looking for domain info acmecorp.com
Tool ACTIVATED: Checking CRM history for jane@acmecorp.com
Tool ACTIVATED: Calculate the Lead Score {"domain_info": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"}, "crm_history": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."}}

--- FINAL AGENT SUMMARY ---
Here's a comprehensive summary of the lead Jane from Acme Corp:

- **Contact Information:**
  - Email: jane@acmecorp.com
  - Company Domain: acmecorp.com

- **Company Information:**
  - Industry: Software/SaaS
  - Size: 501-1000 employees
  - Revenue: $50M - $100M

- **CRM History:**
  - Last Contacted: November 15, 2025
  - Current Status: Cold Lead
  - Notes: Attended a webinar, but there has been no follow-up yet.

- **Lead Score:**
  - Medium

This summarized information indicates that Jane, representing Acme Corp, is from a medium-sized, revenue-generatin